In [1]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Direktori dataset
train_dir = "intel-image-classification/seg_train/seg_train"
val_dir = "intel-image-classification/seg_test/seg_test"

# Augmentasi data
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, zoom_range=0.2, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(train_dir, target_size=(150, 150), 
batch_size=32, class_mode='categorical')
val_generator = val_datagen.flow_from_directory(val_dir, target_size=(150, 150), batch_size=32, 
class_mode='categorical')


Found 14034 images belonging to 6 classes.
Found 3000 images belonging to 6 classes.


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Definisi model CNN
model = Sequential([
Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
MaxPooling2D(2,2),
Conv2D(64, (3,3), activation='relu'),
MaxPooling2D(2,2),
Flatten(),
Dense(128, activation='relu'),
Dropout(0.5),
Dense(6, activation='softmax')
])

# Kompilasi model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Training model
model.fit(train_generator, validation_data=val_generator, epochs=10)

# Simpan model
model.save('cnn_model.h5')

c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 433ms/step - accuracy: 0.4682 - loss: 1.4196

c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


439/439 ━━━━━━━━━━━━━━━━━━━━ 200s 452ms/step - accuracy: 0.4683 - loss: 1.4191 - val_accuracy: 0.6817 - val_loss: 0.8473
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 156s 354ms/step - accuracy: 0.6441 - loss: 0.9545 - val_accuracy: 0.7033 - val_loss: 0.8452
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 186s 424ms/step - accuracy: 0.6775 - loss: 0.8698 - val_accuracy: 0.7573 - val_loss: 0.6902
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 321s 731ms/step - accuracy: 0.7264 - loss: 0.7757 - val_accuracy: 0.7830 - val_loss: 0.6240
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 249s 566ms/step - accuracy: 0.7349 - loss: 0.7394 - val_accuracy: 0.7983 - val_loss: 0.5975
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 229s 521ms/step - accuracy: 0.7477 - loss: 0.6998 - val_accuracy: 0.8110 - val_loss: 0.5574
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 341s 776ms/step - accuracy: 0.7586 - loss: 0.6804 - val_accuracy: 0.8093 - val_loss: 0.5738
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 217s 494ms/step - accuracy: 0.7601 - loss: 0.66

In [4]:
pip install pillow

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [17]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load model yang telah dilatih
model = load_model('cnn_model.h5')

# Load label kelas
class_labels = list(train_generator.class_indices.keys())

# Fungsi untuk adaptive gamma correction
def adjust_gamma(image, gamma=1.0):
    invGamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** invGamma * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

# Fungsi untuk mengukur intensitas cahaya dan klasifikasi

def classify_light_intensity(intensity):
    if intensity < 50:
        return "Gelap"
    elif 50 <= intensity < 150:
        return "Redup"
    else:
        return "Terang"

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Hitung intensitas cahaya untuk koreksi gamma
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mean_intensity = np.mean(gray_frame)
    light_classification = classify_light_intensity(mean_intensity)

    gamma = 2.0 if mean_intensity < 50 else 0.8 if mean_intensity > 200 else 1.0

    # Koreksi gamma
    corrected_frame = adjust_gamma(frame, gamma=gamma)

    # Night Vision mode
    night_vision = cv2.cvtColor(corrected_frame, cv2.COLOR_BGR2GRAY)
    night_vision = cv2.applyColorMap(night_vision, cv2.COLORMAP_JET)

    # Preprocessing gambar
    img = cv2.resize(corrected_frame, (150, 150))
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=0)

    # Prediksi kelas
    pred = model.predict(img)
    label = class_labels[np.argmax(pred)]

    # Tampilkan hasil
    cv2.putText(corrected_frame, f'Class: {label}', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(corrected_frame, f'Gamma: {gamma:.2f}', (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 140, 0), 2)
    cv2.putText(corrected_frame, f'Light Intensity: {mean_intensity:.2f}', (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 215, 0), 2)
    cv2.putText(corrected_frame, f'Lighting: {light_classification}', (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

    # Tampilkan frame dan night vision
    cv2.imshow('Frame (Corrected)', corrected_frame)
    cv2.imshow('Night Vision', night_vision)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━

In [7]:
pip install python-opencv

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement python-opencv (from versions: none)
ERROR: No matching distribution found for python-opencv


In [8]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
